Prácticas del Curso de Modelado y Simulación Topológica, Álvaro Torras Casas (c) 2026, 
Departamento de Matemática Aplicada I, 
Escuela Técnica Superior de Ingeniería Informática, Universidad de Sevilla, España.

# Práctica 1: Homología Simplicial

Vamos a estudiar como calcular homología simplicial sobre un cuerpo mediante algunas librerías de Python. 

Empezamos instalando los módulos necesarios. 
Sobretodo, vamos a tener que usar la librería `Numpy` (https://numpy.org/) de python que sirve para trabajar con matrices.
Por otro lado, para los cálculos con complejos simpliciales y homología necesitaremos utilizar la librería `GUDHI` (https://gudhi.inria.fr/) 

In [ ]:
%%capture
pip install numpy gudhi "sympy>=1.14" tadasets

También utilizaremos las liberías `NetworkX` y `Matplotlib` para visualizar algunos complejos simpliciales.

In [ ]:
%%capture
pip install networkx matplotlib

Por otro lado, vamos a instalar el módulo PHAT (https://www.sciencedirect.com/science/article/pii/S0747717116300098) que nos permitirá trabajar con las matrices sobre $\mathbb{Z}_2$. Para esto utilizaremos una versión algo modificada.

In [ ]:
%%capture
!pip install setuptools pybind11
!pip install --no-build-isolation git+https://bitbucket.org/atorras1618/phat.git

Como seguramente estaremos ejecutando la libreta en Google Colab, ejecutamos la siguiente celda para importar funciones auxiliares para trabajar con los laboratorios.

In [ ]:
%%capture
import os

# Check if we are running in Colab
if 'google.colab' in str(get_ipython()):
    repo_name = 'MyST'
    if not os.path.exists(repo_name):
        !git clone https://github.com/atorras1618/{repo_name}.git
    
    os.chdir(repo_name)
    import sys
    sys.path.append(os.getcwd())

import funciones_auxiliares

# Parte 1: manipulaciones básicas de complejos simpliciales abstractos

En esta parte trabajaremos con el objeto simplex Tree de Gudhi para definir algunos complejos simpliciales abstractos.

### Ejemplo 1: Un complejo sencillo
Vamos a crear un complejo simplicial mediante la estructura de datos `simplex_tree` de GUDHI. Para esto, importaremos el objeto simplex tree con el alias `st` y le añadiremos los símplices del siguiente complejos simplicial $K$:

![Ejemplo 1](images/ejemplo-1.png)

In [ ]:
import gudhi
K = gudhi.SimplexTree()
# Añadimos los símplices de dimensión 0
K.insert([0])
K.insert([1])
K.insert([2])
K.insert([3])
# Añadimos los símplices de dimensión 1
K.insert([0,1])
K.insert([0,2])
K.insert([1,2])
K.insert([1,3])
K.insert([2,3])
# Añadimos el símplice de dimensión 2
K.insert([1,2,3])

Cada vez que insertamos un símplice en el objeto `st`, el método devuelve `True` cuando la inserción se ha realizado con éxito. Por otro lado, podemos comprobar leer todos los símplices que hemos insertado mediante el método `get_simplices()`.

In [ ]:
print(list(K.get_simplices()))

Como podemos combrobar, la lista que nos devuelve GUDHI incluye los valores de filtración de todos los símplices. Como no los hemos configurado, todos toman el valor por defecto `0.0`. En estas prácticas incluimos dos funciones para trabajar con la lista de símplices e imprimirla en pantalla: `diccionario_simplices(K)`y `ver_simplices(K)`



In [ ]:
from funciones_auxiliares import diccionario_simplices, ver_simplices

In [ ]:
dict_spx = diccionario_simplices(K)
for dim in range(3):
    print(f"Símplices en dimensión {dim}:")
    print(dict_spx[dim])

El mismo resultado lo podemos obtener de forma más cómoda mediante `ver_simplices`

In [ ]:
ver_simplices(K)

Vamos ahora a representar el complejo simplicial en el plano. 

In [ ]:
from funciones_auxiliares import plot_simplex_tree_2D

plot_simplex_tree_2D(K, figsize=(4,4))

Podemos asignar posiciones a los nodos y especificarlas a la hora de representar el complejo simplicial:

In [ ]:
pos={0:[0,0],1:[1,0],2:[0,1],3:[1,1]}
plot_simplex_tree_2D(K, pos=pos, figsize=(4,4))

Alternativamente, podemos crear el mismo complejo simplicial añadiendo únicamente los símplices maximales, sin necesidad de añadir sus caras de forma explícita.

In [ ]:
import gudhi
K_aux = gudhi.SimplexTree()
# Añadimos los símplices maximales 
# de dimensión 1
K_aux.insert([0,1])
K_aux.insert([0,2])
# y el de dimensión 2
K_aux.insert([1,2,3])

Y comprobamos que el complejo simplicial resultante es idéntico al anterior.

In [ ]:
ver_simplices(K_aux)
plot_simplex_tree_2D(K_aux, pos=pos, figsize=(4,4))

Nótese que Gudhi guarda los símplices como pares (símplice, valor de filtración). El valor por defecto es `0.0`.

In [ ]:
print(f"El complejo simplicial tiene {K.num_vertices()} vértices y un total de {K.num_simplices()} símplices y su dimensión es {K.dimension()}.")

Podemos calcular la característica de Euler $\chi(K)$ sumando y restando símplices, en este caso obtenemos $\chi(K)=0$

In [ ]:
dict_spx = diccionario_simplices(K)
euler_chi = 0
for dim in range(K.dimension()+1):
    euler_chi += ((-1)**dim) * len(dict_spx[dim])

print(f"La característica de Euler del complejo simplicial es {euler_chi}")

Por comodidad, esta misma función se encuentra en el fichero de funciones auxiliares, lo podemos cargar y ejecutar:

In [ ]:
from funciones_auxiliares import característica_euler

característica_euler(K)

Gudhi también permite calcular los números de Betti del complejo simplicial. Para esto, simplimente ejecutaremos el método `compute_persistence()`seguido de la función que nos devuelve los números de Betti.

In [ ]:
K.compute_persistence()
numeros_Betti = K.betti_numbers()
print(numeros_Betti)

Entonces obtenemos $\beta_0(K)=1$ y $\beta_1(K)=1$, cumpliéndose la fórmula $\chi(K)=\beta_0(K)-\beta_1(K)=0$ (en este caso los números de Betti de dimensiiones $\geq 2$ son ignorados pur GUDHI. Si queremos obtener números de Betti superiores, por ejemplo, hasta dimensión 5, entonces debemos decir a GUDHI que el complejo simplicial tiene dimensión $6$:

In [ ]:
K.set_dimension(6)
K.compute_persistence()
numeros_Betti = K.betti_numbers()
print(numeros_Betti)

Para terminar, volvemos a configurar la dimensión a $2$.

In [ ]:
K.set_dimension(2)

### Ejemplo 2: Triangulación de un tetraedro vacío

Consideramos un tetraedro vacío al que llamaremos `T`. Este lo podemos definir fácilmente, incluyendo primero todo el tetraedro y, seguidamente, substrayendo el símplice maximal mediante el método `remove_maximal_simplex`

In [ ]:
T = gudhi.SimplexTree()
T.insert([0,1,2,3])
T.remove_maximal_simplex([0,1,2,3])

Podemos comprovar que efectivamente `T`es el tetraedro vacío. Nótese que la representación tiene sus limitaciones.

In [ ]:
ver_simplices(T)
plot_simplex_tree_2D(T)

Podemos inspeccionar algunas propiedades de `T`:

In [ ]:
print(f"Número de vértices en T: {T.num_vertices()}")
print(f"Número de símplices en T: {T.num_simplices()}")
print(f"Dimensión de T: {T.dimension()}")
print(f"Característica de Euler de T: {característica_euler(T)}")

Cuando calculamos los números de Betti, vemos que algo no acaba de encajar:

In [ ]:
T.compute_persistence()
print(f"Betti numbers: {T.betti_numbers()}")

En particular, tenemos que $\chi(T)=2 \neq 1 = \beta_0(T) - \beta_1(T)$. En realidad---como ya hemos comentado en el ejemplo anterior---debemos indicar a GUDHI que el complejo con el que estamos tratando tiene una dimensión más. Esto asegura el cálculo de los números de Betti hasta dimensión $2$:

In [ ]:
T.set_dimension(3)
T.compute_persistence()
print(f"Betti numbers: {T.betti_numbers()}")

De esta forma, se comprueba la fórmula conocida: $\chi(T)=\beta_0(T) - \beta_1(T) + \beta_2(T)$

### Ejemplo 3: un toro

Vamos a considerar una triangulación cásica del toro `Toro` que consiste en 16 triángulos como en la siguente figura:

![Ejemplo 1](images/toro.png)

In [ ]:
Toro = gudhi.SimplexTree()

# Define triangles for a torus triangulation
triangles = [
    [0,1,3], [1,3,4], [1,2,4], [2,4,5], [0,2,5], [0,3,5], # primera columna
    [3,4,6], [4,6,7], [4,5,7], [5,7,8], [3,5,8], [3,6,8], # segunda columna
    [0,6,7], [0,1,7], [1,7,8], [1,2,8], [2,6,8], [0,2,6]  # tercera columna
]

# Insert simplices into the tree
for triangle in triangles:
    Toro.insert(triangle)

In [ ]:
ver_simplices(Toro)

Podemos entonces imprimir información relativa al toro. Esta vez ya incluimos que la dimensión debe ser $3$.

In [ ]:
print(f"Number of vertices: {Toro.num_vertices()}")
print(f"Number of simplices: {Toro.num_simplices()}")
Toro.set_dimension(3)
print(f"Dimension: {Toro.dimension()}")

# Compute persistence to verify the Betti numbers
# For a torus, we expect: B0=1, B1=2, B2=1
Toro.compute_persistence()
betti = Toro.betti_numbers()
print(f"Betti numbers: {betti}")

### Ejercicio 1:
Calcular la homología simplicial de:
1. El tetraedro vacío `T`
2. El toro
3. El plano proyectivo

## Parte 2: homología simplicial via la forma normal de Smith



Vamos a calcular la forma normal de smith de algunos de los ejemplos del apartado anterior. Empezamos con el complejo $K$. Para esto, tenemos que calcular primero su matriz diferencial. Empezaremos repasando como calcular las fronteras de sus símplices.

In [ ]:
from funciones_auxiliares import lista_simplices
list_spx = lista_simplices(K)
print(list_spx)

Asumiendo que la orientación de $K$ viene dada por los índices en los vértices, podemos calcular los distintos índices de las caras con respecto a un símplice. Por ejemplo, recordamos que $[1]$ y $[2]$ tienen índices respectivos $-1$ y $1$ con respecto a $[1,2]$.

In [ ]:
spx = [1,2]
print(f"Simplices en la frontera de {spx}")
for idx in range(len(spx)):
    cara = spx[:idx] + spx[idx+1:]
    coef = (-1)**idx
    print(f"cara {cara} con coeficiente {coef:2d}")

Asumiendo que $C_0(K)$ esta generado por los $0$-símplices $\langle [0], [1], [2], [3] \rangle$, el borde $\partial([1,2])=[2]-[1]$ puede expresarse mediante el vector $(0,-1,1,0)$.
Siguendo esta idea, podemos escribir una función que calcula las matrices de los diferenciales del complejo de cadenas:
$$
0 \leftarrow C_0(K) \leftarrow C_1(K) \leftarrow C_2(K) \leftarrow \cdots
$$
A continuación, cargaremos la función `diferenciales` que nos permite obtener una lista con los diferenciales en cada dimensión. Esta función utiliza el paquete de python de álgebra simbólica llamado `sympy`. Una ventaja de este paquete es que nos permite imprimir las matrices en formato matemático. Lo comprovamos con el complejo simplicial $K$.

In [ ]:
from funciones_auxiliares import diferenciales
import sympy as sp

list_dif = diferenciales(K)
for dim in range(K.dimension()+1):
    print(f"Diferencial en dimensión {dim}:")
    print()
    sp.pprint(list_dif[dim])
    print()

De esta forma, podemos llenar las matrices de los diferenciales.
Podemos cargar la función de SymPy que nos permite calcular la descomposición de formas normales de Smith. En particular, la función `smith_normal_decomp`devuelve la forma normal de Smith $S$ junto con las matrices $U$ y $V$ tales que $S = UA V$, donde $A$ es la matriz que queremos transformar a la forma normal de Smith.

In [ ]:
from sympy.matrices.normalforms import smith_normal_decomp

A = list_dif[1]
S, U, V = smith_normal_decomp(A, domain=sp.ZZ)

Seguidamente, comprobamos que efectivamente $UAV=S$

In [ ]:
U*A*V

In [ ]:
S

También podemos obtener el rango de la forma normal de Smith mediante el método `rank()`

In [ ]:
S.rank()

Podemos calcular las formas normales de smith para el diferencial en cada dimensión y utilizarlas para calcular los números de Betti:

In [ ]:
K.num_vertices()

In [ ]:
S.shape[1]

Podemos calcular los números de Betti a partir de la forma Normal de Smith mediante la fórmula $\beta_d = z_d - b_d$ vista en la asignatura.

In [ ]:
# el rango en dimensión 0 es igual al número de 0-símplices/vértices
z_dim = K.num_vertices()
for dim in range(K.dimension()):
    A = list_dif[dim+1]
    S, _, _ = smith_normal_decomp(A, domain=sp.ZZ)
    b_dim = S.rank()
    print(f"Dimensión {dim}:")
    print(f"z_{dim} = {z_dim}, b_{dim} = {b_dim}")
    print(f"betti_{dim} = z_{dim} - b_{dim} = {z_dim-b_dim}")
    print("--------------------------------")
    z_dim = S.shape[1]-b_dim

Y comprobamos que coincide efectivamente con el cálculo realizado con GUDHI.

In [ ]:
K.compute_persistence()
betti_numbers = K.betti_numbers()
betti_numbers

### Ejercicio 2:

Volver a calcular los grupos de homología del Ejercicio 1 mediante la forma normal de Smith. ¿Coinciden con los del ejercicio anterior?

## Parte 3: complejos simpliciales a partir de nubes de puntos

Para nubes de puntos existe un paquete muy útil llamado tdasets (https://tadasets.scikit-tda.org).

Lo cargamos y lo utilizamos para calcular una muestra de puntos $X$ cerca de un círculo de radio $1$.

In [ ]:
import tadasets

circle_points = tadasets.dsphere(n=70, d=1, r=1.0, noise=0.2, seed=5)

Podemos visualizar la muestra mediante el método `scatter`de Matplotlib:

In [ ]:
import matplotlib.pyplot as plt
# Coordenadas x e y de los puntos
x = circle_points[:,0]
y = circle_points[:,1]
plt.scatter(x, y, color="black")
plt.gca().set_aspect("equal")
plt.show()

También podemos utilizar `tadasets` para generar una nube de puntos alrededor de un toro inmerso en $\mathbb{R}^3$. Tomamos una muestra de $n$ puntos de un toro de radio mayor $c$ y radio menor $a$.

In [ ]:
import numpy as np

puntos_toro = tadasets.torus(n=2000, c=2.0, a=1.0)

Podemos también usar matplotlib para visualizar los puntos tomados del toro:

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')
# Extract X, Y, and Z coordinates for plotting
x = puntos_toro[:, 0]
y = puntos_toro[:, 1]
z = puntos_toro[:, 2]
# Scatter plot: 's' is point size, 'c' assigns colors based on Z height
ax.scatter(x, y, z, s=5, c=z, cmap='plasma_r', alpha=0.8)
# Set the aspect ratio to be equal 
ax.set_box_aspect((np.ptp(x), np.ptp(y), np.ptp(z)))
# Optional: clean up the axes
ax.set_title("3D Point Cloud of a Torus", fontsize=20)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
plt.show()

### El complejo de Vietoris-Rips

El complejo de Vietoris-Rips permite calcular un complejo simplicial a partir de un espacio métrico finito. Básicamente, los vértices son los puntos subyacentes y las aristas se incluyen cuando su distancia es menor que un valor dado.

Vamos a calcular el complejo de Vietoris-Rips para los ejemplos anteriores.

In [ ]:
maxima_distancia = 0.5
rips_circle = gudhi.RipsComplex(points=circle_points, max_edge_length=maxima_distancia)

Con esta operación hemos inicializado el objeto `RipsComplex`de Gudhi que nos permitirá calcular el complejo de Vietoris-Rips hasta el valor máximo de filtración. Seguidamente, calculamos su complejo simplicial y lo visualizamos.

In [ ]:
st_circle = rips_circle.create_simplex_tree()
plot_simplex_tree_2D(st_circle, pos=circle_points, with_labels=False, node_size=30, points=True)

Parece que algo no va bien... Hay una serie de triángulos que deberían aparecer pero no se visualizan en la figura anterior. Esto es porque, por defecto, el complejo simplicial creado por Gudhi a partir de un complejo de Vietoris-Rips es de dimensión $1$, como podemos comprobar: 

In [ ]:
st_circle.dimension()

Esto es así ya que el cálculo explícito del complejo de Vietoris-Rips es muy costoso. Si queremos que nuestro complejo tenga todos los cliques posibles hasta dimensión $2$, podemos conseguirlo fácilmente mediante el método `expansion()`.

In [ ]:
st_circle.expansion(2)
plot_simplex_tree_2D(st_circle, pos=circle_points, with_labels=False, node_size=30, points=True)

Como antes, podemos obtener información sobre el complejo simplicial resultante: 

In [ ]:
print("Información sobre el complejo simplicial:")
print(f"Dimensión: {st_circle.dimension()}")
print(f"Número de vértices: {st_circle.num_vertices()}")
print(f"Número de símplices: {st_circle.num_simplices()}")
st_circle.compute_persistence()
print(f"Números de Betti: {st_circle.betti_numbers()}")

Vamos ahora a calcular el complejo de Vietoris-Rips para la muestra de puntos del toro que hemos tomado antes. Calcularemos el complejo hasta un valor máximo de filtración y lo visualizaremos en tres dimensiones mediante una función auxiliar: `plot_simplex_tree_3D`

In [ ]:
rips_toro = gudhi.RipsComplex(points=puntos_toro, max_edge_length=0.5)
st_toro = rips_toro.create_simplex_tree()
st_toro.expansion(3)

In [ ]:
import matplotlib.pyplot as plt
from funciones_auxiliares import plot_simplex_tree_3D

plot_simplex_tree_3D(st_toro, puntos_toro)
plt.tight_layout()
plt.show()

Como vemos, el toro aún tiene bastantes agujeros. Vamos a imprimir información sobre el complejo simplicial:

In [ ]:
print("Información sobre el complejo simplicial del toro:")
print(f"Dimensión: {st_toro.dimension()}")
print(f"Número de vértices: {st_toro.num_vertices()}")
print(f"Número de símplices: {st_toro.num_simplices()}")
st_toro.compute_persistence()
print(f"Números de Betti: {st_toro.betti_numbers()}")

### El complejo Alfa

Como hemos visto, los complejos de Vietoris-Rips pueden incrementar su número de símplices de forma bastante rápida. Vamos ahora a considerar el complejo Alfa, que tiene un número de simplices controlado para dimensiones de ambiente pequeñas.

Además, resulta ser que el complejo alfa de radio $r$ es homotópicamente equivalente a la unión de bolas cerradas centradas en sus vértices de radio $r$.

Empezaremos creando el complejo de alfa de los puntos tomados alrededor del círculo donde el máximo cuadrado de radio de filtración es $\leq 0.2$

In [ ]:
ac_circle = gudhi.AlphaComplex(points=circle_points)
st_ac_circle = ac_circle.create_simplex_tree(max_alpha_square=0.2)

In [ ]:
plot_simplex_tree_2D(st_ac_circle, pos=circle_points, with_labels=False, node_size=30, points=True)

Como podemos comprobar, el complejo de alfa ha capturado el ciclo que se encuentra dentro de la muestra de datos. También podemos ver que este complejo utiliza muchos menos símplices que el complejo de Vietoris-Rips y parece ser que contiene menos `ruido`

In [ ]:
print("Información sobre el complejo simplicial:")
print(f"Dimensión: {st_ac_circle.dimension()}")
print(f"Número de vértices: {st_ac_circle.num_vertices()}")
print(f"Número de símplices: {st_ac_circle.num_simplices()}")
st_ac_circle.compute_persistence()
print(f"Números de Betti: {st_ac_circle.betti_numbers()}")

Vemos que lo mismo ocurre con el complejo simplicial tomado a partir de la muestra del toro. Tomando el complejo de alfa hasta un máximo cuadrado de filtración $0.3$, acabamos recuperando los números de Betti del toro.

In [ ]:
ac_toro = gudhi.AlphaComplex(points=puntos_toro)
st_ac_toro = ac_toro.create_simplex_tree(max_alpha_square=0.3)
st_ac_toro.expansion(4)

In [ ]:
import matplotlib.pyplot as plt
from funciones_auxiliares import plot_simplex_tree_3D

plot_simplex_tree_3D(st_ac_toro, puntos_toro)
plt.tight_layout()
plt.show()

In [ ]:
print("Información sobre el complejo simplicial:")
print(f"Dimensión: {st_ac_toro.dimension()}")
print(f"Número de vértices: {st_ac_toro.num_vertices()}")
print(f"Número de símplices: {st_ac_toro.num_simplices()}")
st_ac_toro.compute_persistence()
print(f"Números de Betti: {st_ac_toro.betti_numbers()}")

### Ejercicio 3:
Calcular muestras suficientemente densas, así como sus respectivas triangulaciones de Vietoris-Rips y Alpha, de los siguientes espacios:
1. El símbolo infinito (mediante la función `infty_sign` de `tadasets`
2. La esfera 3 dimensional (mediante la función `dsphere` de `tadasets`